# Single-card gpt-oss-120b development runtime
Open this notebook in Colab **only on an already provisioned, trusted Linux runtime**. Do not purchase a GPU to run the checks. Use one RTX PRO 6000 Blackwell 96 GB. The client runs inside the same private runtime; no external ingress or port sharing is needed. Use a private notebook and do not share the runtime. See `docs/setup/gpt-oss.md`.

Checkpoint recovery requires a writable **persistent filesystem** with 75 GiB free (e.g. an attached volume on a Colab local runtime). Set `CACHE` below to its mount point. For managed Colab, mount your private Drive manually and use `/content/drive/MyDrive/gpt-oss-cache` with sufficient quota. Do not use temporary `/content` for recovery across VM replacement. Replace a lost runtime by reattaching the same volume and rerunning from the top; completed Hub blobs are reused and partial downloads can resume. This recovers weights, not live requests or KV cache.

In [ ]:
import os, json, shutil, subprocess, sys, time, secrets
from pathlib import Path
MODEL = 'openai/gpt-oss-120b'
REV = 'b5c939de8f754692c1647ca79fbf85e8c1e70f8a'
CACHE = Path('/mnt/gpt-oss-cache')
LIMIT, OUTPUT, SLOTS = 8192, 1024, 1  # 7168 prompt tokens including Harmony framing, tools and history

def check_gpu(csv):
    rows = csv.strip().splitlines()
    if len(rows) != 1: raise RuntimeError('Exactly one GPU required')
    name, total, free, cc, driver = map(str.strip, rows[0].split(','))
    if not all(x in name for x in ('RTX PRO 6000','Blackwell')) or cc != '12.0' or int(total)<90000 or int(free)<88000:
        raise RuntimeError('Incompatible or busy GPU: '+rows[0])
    return name, driver

In [ ]:
gpu = check_gpu(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.free,compute_cap,driver_version','--format=csv,noheader,nounits'],text=True))
print('GPU / driver:',gpu)
if not CACHE.is_absolute() or not CACHE.is_dir() or CACHE.stat().st_dev == Path('/').stat().st_dev:
    raise RuntimeError('Choose an existing directory on a separate persistent mounted volume')
used = sum(p.stat().st_size for p in (CACHE/'models--openai--gpt-oss-120b'/'blobs').glob('*') if p.is_file())
if shutil.disk_usage(CACHE).free + min(used,65248815744) < 75*2**30: raise RuntimeError('Insufficient checkpoint cache space')

In [ ]:
# Isolated vLLM environment; do not replace the running notebook kernel's torch.
ENV = Path('/content/gpt-oss-env'); ENV.parent.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','venv',str(ENV)],check=True)
PY = str(ENV/'bin/python'); CLI = str(ENV/'bin/vllm')
subprocess.run([PY,'-m','pip','install','vllm==0.15.1'],check=True)
subprocess.run([CLI,'--version'],check=True)

In [ ]:
fetch = """from huggingface_hub import snapshot_download
import sys
print(snapshot_download(sys.argv[1],revision=sys.argv[2],cache_dir=sys.argv[3],allow_patterns=['*.json','*.jinja','*.txt','*.model','*.tiktoken','model-*.safetensors'],ignore_patterns=['original/*']))"""
weights = Path(subprocess.check_output([PY,'-c',fetch,MODEL,REV,str(CACHE)],text=True).strip().splitlines()[-1])
assert weights.name == REV and json.loads((weights/'config.json').read_text())['quantization_config']['quant_method']=='mxfp4'
index=json.loads((weights/'model.safetensors.index.json').read_text())
assert all((weights/f).is_file() for f in set(index['weight_map'].values()))

In [ ]:
import requests, socket
if 'server' in globals() and server.poll() is None: raise RuntimeError('Stop existing server first')
with socket.socket() as probe: probe.bind(('127.0.0.1',8000))
key = secrets.token_urlsafe(32)
client = requests.Session(); client.trust_env = False
base = 'http://127.0.0.1:8000/v1'; auth = {'Authorization':'Bearer '+key}
args = [CLI,'serve',str(weights),'--served-model-name',MODEL,'--host','127.0.0.1','--port','8000','--tensor-parallel-size','1','--max-model-len',str(LIMIT),'--max-num-seqs',str(SLOTS),'--tool-call-parser','openai','--enable-auto-tool-choice']
log = Path('/content/gpt-oss-server.log')
with log.open('w') as out:
    server = subprocess.Popen(args,env={**os.environ,'VLLM_API_KEY':key},stdout=out,stderr=subprocess.STDOUT,start_new_session=True)
import signal
def stop():
    if server.poll() is None:
        os.killpg(server.pid,signal.SIGTERM)
        try: server.wait(30)
        except subprocess.TimeoutExpired: os.killpg(server.pid,signal.SIGKILL); server.wait()
try:
    for _ in range(600):
        if server.poll() is not None: raise RuntimeError(f'Startup failed; inspect {log} privately. Check driver/wheel SM120 compatibility; do not fall back to BF16.')
        try: r=client.get(base+'/models',timeout=2)
        except requests.ConnectionError: time.sleep(2); continue
        if r.status_code==503: time.sleep(2); continue
        assert r.status_code==401, 'Unauthenticated access was not rejected'
        r=client.get(base+'/models',headers=auth,timeout=5); r.raise_for_status()
        assert MODEL in [m['id'] for m in r.json()['data']]; break
    else: raise TimeoutError('Server startup timed out')
except BaseException:
    stop(); raise

In [ ]:
tool = {'type':'function','function':{'name':'lookup_item','description':'Return the inventory name for an item ID','parameters':{'type':'object','properties':{'id':{'type':'string'}},'required':['id'],'additionalProperties':False}}}
def chat(messages):
    r=client.post(base+'/chat/completions',headers=auth,json=dict(model=MODEL,messages=messages,tools=[tool],tool_choice='auto',temperature=0,max_tokens=OUTPUT),timeout=180)
    r.raise_for_status(); return r.json()['choices'][0]['message']
messages=[{'role':'user','content':'Look up inventory item A17 with lookup_item and tell me its exact name.'}]
a=chat(messages); calls=a.get('tool_calls') or []; assert len(calls)==1
c=calls[0]; assert c['id'] and c['type']=='function' and c['function']['name']=='lookup_item' and json.loads(c['function']['arguments'])=={'id':'A17'}
name='Orchid-'+secrets.token_hex(4)
messages.extend([{'role':'assistant','content':a.get('content'),'tool_calls':calls},{'role':'tool','tool_call_id':c['id'],'content':json.dumps({'name':name})}]); answer=chat(messages)
assert not answer.get('tool_calls') and name in (answer.get('content') or '')
plain=chat([{'role':'user','content':'Say hello. No lookup is needed.'}]); assert not plain.get('tool_calls') and plain.get('content')
print('PASS: structured lookup, result round trip, and no-tool control',gpu)

## Stop and recover
Run the next cell when finished. For in-notebook development, use `client`, `base`, and `auth` before stopping; never print `auth`. If initialization fails, inspect the private log, verify the driver supports the installed CUDA wheel, and rerun after fixing it. The first launch also needs outbound network access for Harmony encodings. On a fresh kernel in the same VM stop any prior server before restarting. Disconnect the already-provisioned runtime separately when done.

In [ ]:
stop(); client.close()